<a href="https://colab.research.google.com/github/Tashfi12/Natural-scene-classification-CNN/blob/main/Image_analysis_with_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Exercise - CNN

This is image data of Natural Scenes around the world. This data contains  images of size 150x150 distributed under 6 categories.

Dataset link: classification.zip

0 - buildings

1 - forest

2 - glacier

3 - mountain

4 - sea

5 - street

The Train, Test and Prediction data is separated in each folder.
Conduct the analysis using an appropriate model and improve the model performance.

In [ ]:
#import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import tensorflow as tf

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!unzip "/content/drive/My Drive/Colab Notebooks/Datasets/classification.zip" -d "/content/drive/My Drive/Colab Notebooks/Datasets/classification"

Archive:  /content/drive/My Drive/Colab Notebooks/Datasets/classification.zip
replace /content/drive/My Drive/Colab Notebooks/Datasets/classification/seg_pred/seg_pred/10004.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
path = '/content/drive/My Drive/Colab Notebooks/Datasets/classification'

In [ ]:
# Define image parameters
img_height, img_width = 150, 150
batch_size = 32

# Load training data
train_dir = path + '/seg_train/seg_train'
train_data = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    labels='inferred',
    label_mode='categorical', # There are 6 categories
    image_size=(img_height, img_width),
    interpolation='nearest',
    batch_size=batch_size,
    shuffle=True
)

# Display some information about the loaded dataset
print("Training dataset loaded successfully.")
print(f"Number of batches: {len(train_data)}")
for images, labels in train_data.take(1):
    print(f"Image batch shape: {images.shape}")
    print(f"Label batch shape: {labels.shape}")
    print(f"Class names: {train_data.class_names}")

Found 14034 files belonging to 6 classes.
Training dataset loaded successfully.
Number of batches: 439
Image batch shape: (32, 150, 150, 3)
Label batch shape: (32, 6)
Class names: ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']


In [ ]:
# Load test data
test_dir = path + '/seg_test/seg_test'
test_data = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    labels='inferred',
    label_mode='categorical',
    image_size=(img_height, img_width),
    interpolation='nearest',
    batch_size=batch_size,
    shuffle=False
)

print("Test dataset loaded successfully.")
print(f"Number of batches: {len(test_data)}")
for images, labels in test_data.take(1):
    print(f"Image batch shape: {images.shape}")
    print(f"Label batch shape: {labels.shape}")
    print(f"Class names: {test_data.class_names}")

Found 3000 files belonging to 6 classes.
Test dataset loaded successfully.
Number of batches: 94
Image batch shape: (32, 150, 150, 3)
Label batch shape: (32, 6)
Class names: ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']


In [ ]:
# Load prediction data
pred_dir = path + '/seg_pred/seg_pred'
pred_data = tf.keras.utils.image_dataset_from_directory(
    pred_dir,
    labels=None,
    label_mode=None,
    image_size=(img_height, img_width),
    interpolation='nearest',
    batch_size=batch_size,
    shuffle=False
)

print("Prediction dataset loaded successfully.")
print(f"Number of batches: {len(pred_data)}")
for images in pred_data.take(1):
    print(f"Image batch shape: {images.shape}")

Found 7301 files.
Prediction dataset loaded successfully.
Number of batches: 229
Image batch shape: (32, 150, 150, 3)


In [ ]:
# Normalize pixel values to be between 0 and 1
def normalize_img(image, label):
    return tf.cast(image, tf.float32) / 255., label

train_data_normalized = train_data.map(normalize_img)
test_data_normalized = test_data.map(normalize_img)
pred_data_normalized = pred_data.map(lambda image: tf.cast(image, tf.float32) / 255.)

print("Datasets normalized.")

Datasets normalized.


In [ ]:
# Build the CNN model
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(img_height, img_width, 3)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(len(train_data.class_names), activation='softmax') # Output layer with 6 classes
])

# Compile the model
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()
print("CNN model built and compiled.")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 148, 148, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 74, 74, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 72, 72, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 36, 36, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 34, 34, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 17, 17, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 36992)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     4,735,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,829,126 (18.42 MB)

 Trainable params: 4,829,126 (18.42 MB)

 Non-trainable params: 0 (0.00 B)

CNN model built and compiled.


In [ ]:
# Train the model
history = model.fit(
    train_data_normalized,
    epochs=10,
    validation_data=test_data_normalized
)

print("Model training complete.")

Epoch 1/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 2385s 5s/step - accuracy: 0.6330 - loss: 0.9381 - val_accuracy: 0.7507 - val_loss: 0.6862
Epoch 2/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 719s 2s/step - accuracy: 0.7776 - loss: 0.6108 - val_accuracy: 0.8063 - val_loss: 0.5699
Epoch 3/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 754s 2s/step - accuracy: 0.8352 - loss: 0.4566 - val_accuracy: 0.8153 - val_loss: 0.5401
Epoch 4/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 748s 2s/step - accuracy: 0.8803 - loss: 0.3332 - val_accuracy: 0.7957 - val_loss: 0.6127
Epoch 5/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 751s 2s/step - accuracy: 0.9173 - loss: 0.2363 - val_accuracy: 0.7830 - val_loss: 0.7196
Epoch 6/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 711s 2s/step - accuracy: 0.9471 - loss: 0.1509 - val_accuracy: 0.7983 - val_loss: 0.8416
Epoch 7/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 725s 2s/step - accuracy: 0.9590 - loss: 0.1192 - val_accuracy: 0.7993 - val_loss: 0.8560
Epoch 8/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 751s 2s/step - accuracy: 0.9719 - loss: 0.0833 - val_acc

In [ ]:
# Evaluate the model on the test data
loss, accuracy = model.evaluate(test_data_normalized)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

94/94 ━━━━━━━━━━━━━━━━━━━━ 47s 502ms/step - accuracy: 0.8030 - loss: 1.1814
Test Loss: 1.1814
Test Accuracy: 0.8030


In [ ]:
# Make predictions on the prediction data
predictions = model.predict(pred_data_normalized)

# For multiclass classification.
predicted_class_indices = np.argmax(predictions, axis=1)

print("Predictions made for the prediction dataset.")
print(f"First 10 predicted class indices: {predicted_class_indices[:10]}")

# Map indices back to class names if needed
class_names = train_data.class_names
predicted_class_labels = [class_names[idx] for idx in predicted_class_indices[:10]]
print(f"First 10 predicted class labels: {predicted_class_labels}")

229/229 ━━━━━━━━━━━━━━━━━━━━ 1046s 5s/step
Predictions made for the prediction dataset.
First 10 predicted class indices: [0 4 5 2 4 1 4 2 4 5]
First 10 predicted class labels: ['buildings', 'sea', 'street', 'glacier', 'sea', 'forest', 'sea', 'glacier', 'sea', 'street']
